# 01 — Reprodução: detecção de mixers na Ethereum

Notebook didático de reprodução do projeto inspirado no artigo *Uma Proposta de Algoritmo para a Detecção de Mixers na Ethereum* (SBSeg 2025).

**Escopo desta versão:** utilizar datasets processados e auditados, verificar integridade, validar e treinar Random Forest, e avaliar o dia 29/10/2020. **Não reprocessa dumps brutos** e não deve ser apresentado como validação cega. Código versionado no [GitHub](https://github.com/BorgesGusta/ethereum-mixer-detection/tree/main/src/ethmix); esta cópia é para execução no Colab.

**Unidade:** janela contrato-destinatário/dia UTC. Um endereço pode produzir várias janelas. Conjunto de treino legado tem 23 positivas de **um único pool** Tornado Cash 10 ETH em março/2025 e 100 negativas selecionadas por inspeção; isso limita a generalização.
**Dados oficiais:** Projeto_Mixers_Ethereum no seu Drive (não o atalho do Bruno).

**Acesso aos dados:** pelo ID de cada arquivo do seu Drive; não exige que a pasta apareça dentro de `MyDrive`.


## 1 — Conceitos e variáveis

| Feature | O que mede |
|---|---|
| `value_entropy` | Diversidade dos valores transferidos |
| `fee_entropy` | Diversidade das taxas |
| `internal_value_entropy` | Diversidade dos valores internos |
| `call_count_mean` | Média das chamadas da transação |

As entropias são Shannon base 2, depois de discretização por arredondamento (4 casas valores, 6 taxas). Parâmetros de engenharia deste projeto não são automaticamente parâmetros explícitos do artigo.

## 2 — Carregar o código diretamente do **seu** Google Drive

A pasta `Projeto_Mixers_Ethereum` está no Drive, mas pode não aparecer na montagem `/content/drive/MyDrive` (por exemplo, se estiver em um local compartilhado). **Não precisamos depender desse caminho.**

Esta célula solicita o login **Google** do Colab (não é token GitHub), acessa **somente os três módulos Python já copiados para o seu projeto**, pelos respectivos IDs do Drive, e confere SHA-256 para garantir que o código corresponde à versão registrada no GitHub.

Os módulos são baixados para a pasta temporária `/content/ethmix_reproducao/src/ethmix/`. Não alteramos dados da pasta compartilhada do Bruno.

In [ ]:
from pathlib import Path
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import io, json, sys, hashlib

# Autoriza somente a conta Google no Colab. Não precisa montar o Meu Drive.
auth.authenticate_user()
creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
drive_api = build('drive', 'v3', credentials=creds, cache_discovery=False)

REPO = 'BorgesGusta/ethereum-mixer-detection'
CODIGO_REFERENCIA = 'src/ethmix (GitHub, hash SHA-256 dos módulos)'
BASE = Path('/content/ethmix_reproducao')  # cache de execução, NÃO o Drive compartilhado

# IDs dos módulos copiados para o seu Projeto_Mixers_Ethereum/src/ethmix.
MODULOS_DRIVE = {
    '__init__.py': '1DnOyPTb_N_GztcgZ5BDp28vp5utfwBph',
    'features.py': '1rxO8fSvW5GABo7YmYVJtcuTTmMvnBQzL',
    'models.py': '10x_kaEy6pHRRYULoud5lFvFZLqGYy6OT',
}
EXPECTED_MODULE_SHA256 = {
    '__init__.py': 'dccb63bbe5fbde892d3059dab7e3efc03f2e0e169d93afa67e8d983e996cd1ae',
    'features.py': 'b75d65bd2326169e6fc52f86f5030337d6e6813df607bab0a9c3cb20922fd0b5',
    'models.py': '7f92d88cafcfc7c939f8c84e4fcfb34a0aa98c0890a86333c294b23f1508d6d9',
}

def obter_arquivo_drive(file_id, destino):
    """Baixa arquivo binário do próprio Drive por ID, sem precisar de um caminho montado."""
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    request = drive_api.files().get_media(fileId=file_id, supportsAllDrives=True)
    with destino.open('wb') as f:
        downloader = MediaIoBaseDownload(f, request, chunksize=4 * 1024 * 1024)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return destino

for nome, file_id in MODULOS_DRIVE.items():
    arquivo = obter_arquivo_drive(file_id, BASE / 'src' / 'ethmix' / nome)
    obtido = hashlib.sha256(arquivo.read_bytes()).hexdigest()
    if obtido != EXPECTED_MODULE_SHA256[nome]:
        raise ValueError(f'SHA-256 diferente para {nome}: {obtido}. Verifique a sincronização GitHub/Drive.')

sys.path.insert(0, str(BASE / 'src'))
# Evita módulos antigos da memória se uma versão anterior do notebook já foi executada.
for nome in ('ethmix.models', 'ethmix.features', 'ethmix'):
    sys.modules.pop(nome, None)
from ethmix.features import FEATURES, DECIMAIS, preparar, calcular_features
from ethmix.models import novo_rf, validar_cv, reamostrar
print('Código carregado do seu Drive (por ID), sem GitHub Token.')
print('Integridade SHA-256 dos três módulos: verificada.')

## 3 — Baixar os dados processados **da sua pasta** e conferir a integridade

Os dois arquivos Parquet já foram copiados para `Projeto_Mixers_Ethereum/data/processed/`. O Colab os obtém **pelos IDs das cópias no seu Drive**, independentemente de onde a pasta aparece na interface. Conferimos seus SHA-256 com o inventário auditado.

Os downloads são temporários. **Não altera nem sobrescreve os datasets originais.**

In [ ]:
import pandas as pd, numpy as np
from datetime import datetime, timezone

# Arquivos COPIADOS para seu próprio projeto, não arquivos da pasta do Bruno.
DADOS_DRIVE = {
    'treino': '1jUHAq6H9FKSgptv4nsCoOmFILH4Py_2S',
    'teste2020': '1g22eCccxC4v_S2RmxPhFZyzfF1ARuzPk',
}
INPUTS = {
    'treino': BASE / 'data' / 'processed' / 'features_treino.parquet',
    'teste2020': BASE / 'data' / 'processed' / 'features_teste_20201029.parquet',
}
HASHES_ESPERADOS = {
    'treino': '0a8baa11cee97d338d7a8d92b984f38192ba1a10ec915cc4a06d67ec94ad5c32',
    'teste2020': '6f8fd4e83e816c785d449116699065ad77425c9da6e7b19d9bcda36f42bddb51',
}

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for bloco in iter(lambda: f.read(4 * 1024 * 1024), b''):
            h.update(bloco)
    return h.hexdigest()

for chave, file_id in DADOS_DRIVE.items():
    arquivo = obter_arquivo_drive(file_id, INPUTS[chave])
    obtido = sha256(arquivo)
    print(chave, arquivo.name, 'SHA-256:', obtido)
    if obtido != HASHES_ESPERADOS[chave]:
        raise ValueError(f'Arquivo {chave} divergente. Não continuar sem auditoria.')
print('Dados do seu Drive: integridade SHA-256 verificada.')

## 4 — Inspecionar as amostras

**Treino (legado):** 123 janelas, 23 positivas e 100 negativas, com positivos de um único contrato. A validação cruzada estratificada por linha não comprova generalização entre contratos.

**Teste (29/10/2020):** apenas contratos com pelo menos 10 transações entram na classificação. Os demais endereços não são negativos verificados.

In [ ]:
treino=pd.read_parquet(INPUTS["treino"])
teste=pd.read_parquet(INPUTS["teste2020"])
MIN_TX=10
SEED=42
assert set(FEATURES).issubset(treino.columns) and set(FEATURES).issubset(teste.columns)
assert {"label","n_tx"}.issubset(treino.columns)
assert {"recipient","n_tx","tornado"}.issubset(teste.columns)
assert set(treino.label.dropna().unique())=={0,1}
treino_e=treino.loc[treino.n_tx>=MIN_TX].copy()
teste_e=teste.loc[teste.n_tx>=MIN_TX].copy()
print("Treino completo:",len(treino),"classes:",treino.label.value_counts().to_dict())
print("Treino elegível:",len(treino_e),"classes:",treino_e.label.value_counts().to_dict())
print("Teste 2020:",len(teste),"janelas; elegíveis:",len(teste_e))
print("Pools de referência:",int(teste_e.tornado.notna().sum()))
display(treino_e.groupby("label")[list(FEATURES)].median().round(3))

## 5 — Validar e treinar o classificador

Random Forest com 100 árvores, profundidade máxima 5, pesos balanceados e semente 42. Oversampling apenas dentro dos folds de treino. As métricas são **internas e exploratórias**, não evidência de teste independente.

In [ ]:
cv=validar_cv(treino_e,seed=SEED,n_splits=5)
display(cv.round(3))
display(cv[["precision","recall","f1"]].agg(["mean","std"]).round(3))
Xt,yt=reamostrar(treino_e[list(FEATURES)],treino_e.label.astype(int),SEED)
modelo=novo_rf(SEED).fit(Xt,yt)
print("RF treinada",modelo.n_estimators,"árvores; max_depth",modelo.max_depth)

## 6 — Aplicar ao dia histórico de 2020

Testamos limiares 0,50 e 0,85 **apenas para caracterização exploratória**. Os demais alertas são endereços sem rótulos confirmados; não podem ser chamados automaticamente de falsos positivos.

A referência de 63 endereços no artigo é um ponto de comparação, **não** um alvo de otimização.

In [ ]:
prob=modelo.predict_proba(teste_e[list(FEATURES)])[:,1]
rows=[]
for limiar in [0.50,0.85]:
    mask=prob>limiar
    recuperados=int(teste_e.loc[mask,"tornado"].notna().sum())
    rows.append({"limiar":limiar,"enderecos_sinalizados":int(mask.sum()),"pools_referencia_recuperados":recuperados,"outros_sinalizados_sem_rotulo_confirmado":int(mask.sum())-recuperados})
resumo=pd.DataFrame(rows)
display(resumo)
print("Confrontar estes números com os relatórios originais antes de afirmar paridade")

## 7 — Registrar a execução **no seu Drive**, sem sobrescrever o legado

Os relatórios são primeiro preparados em `/content/ethmix_reproducao/results/`, depois **enviados por API do Google Drive** para a pasta `Projeto_Mixers_Ethereum/results/`, usando seu ID. Isso funciona mesmo que o projeto não esteja montado em `MyDrive`.

Cada execução cria uma pasta exclusiva em `results/reproducao_consolidada/AAAA...Z/`, com CSVs e manifesto. Os modelos e dados originais não são sobrescritos.

**Atenção:** os resultados ainda precisam ser confrontados com o legado; não se trata de validação cega.

In [ ]:
ts = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT = BASE / 'results' / 'reproducao_consolidada' / ts
OUT.mkdir(parents=True, exist_ok=False)
cv.to_csv(OUT / 'cv_folds.csv', index=False)
resumo.to_csv(OUT / 'resumo_2020.csv', index=False)
manifesto = {
    'experimento': 'reproducao_consolidada_01',
    'data_utc': ts,
    'repo': REPO,
    'codigo_ref': CODIGO_REFERENCIA,
    'codigo_sha256': EXPECTED_MODULE_SHA256,
    'arquivos_drive_ids': {'modulos': MODULOS_DRIVE, 'dados': DADOS_DRIVE},
    'dados_sha256': HASHES_ESPERADOS,
    'min_tx': MIN_TX, 'seed': SEED,
    'rf': {'n_estimators': 100, 'max_depth': 5, 'class_weight': 'balanced'},
    'limiares_exploratorios': [0.50, 0.85],
    'ressalvas': ['positivos de um unico contrato',
                 'enderecos nao rotulados nao sao negativos',
                 'teste 2020 previamente explorado']
}
(OUT / 'manifesto.json').write_text(
    json.dumps(manifesto, indent=2, ensure_ascii=False), encoding='utf-8')

# IDs das pastas que pertencem ao seu Projeto_Mixers_Ethereum.
PASTA_RESULTADOS_ID = '1iZKAeZPEb6JOlH8KfqsR6-wqI1ygmojI'

def obter_ou_criar_pasta(nome, parent_id):
    seguro = nome.replace("'", "\'")
    query = (f"name = '{seguro}' and '{parent_id}' in parents and "
             "mimeType = 'application/vnd.google-apps.folder' and trashed = false")
    encontrados = drive_api.files().list(
        q=query, fields='files(id,name),nextPageToken', supportsAllDrives=True,
        includeItemsFromAllDrives=True, pageSize=100).execute().get('files', [])
    if encontrados:
        return encontrados[0]['id']
    nova = drive_api.files().create(body={
        'name': nome, 'mimeType': 'application/vnd.google-apps.folder',
        'parents': [parent_id]}, fields='id').execute()
    return nova['id']

pasta_experimento = obter_ou_criar_pasta('reproducao_consolidada', PASTA_RESULTADOS_ID)
pasta_execucao = obter_ou_criar_pasta(ts, pasta_experimento)
for nome in ('cv_folds.csv', 'resumo_2020.csv', 'manifesto.json'):
    arquivo = OUT / nome
    tipo = 'application/json' if nome.endswith('.json') else 'text/csv'
    media = MediaFileUpload(str(arquivo), mimetype=tipo, resumable=True)
    drive_api.files().create(body={'name': nome, 'parents': [pasta_execucao]},
                             media_body=media, fields='id').execute()
print('Resultados registrados no seu Drive:')
print('https://drive.google.com/drive/folders/' + pasta_execucao)